# FiremeX — train the fire model on D-Fire (Google Colab)

Trains the model the FiremeX add-on runs, on **D-Fire**: 21,527 real photos of
fire and smoke. Everything happens in Colab, so nothing large touches your
computer; at the end you download one file of about 6 MB.

1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. **Runtime → Run all.** About 1½–2 hours. Keep the tab open.
3. The last cell downloads `fire_model_dfire.pt`. Give it to FiremeX.

Dataset: D-Fire by Gaia (github.com/gaiasd/DFireDataset) — please credit it.

In [ ]:
# The add-on runs Ultralytics 8.2.0. Training with the same version means the
# saved model loads in the add-on exactly as it does here.
!pip -q install ultralytics==8.2.0 kagglehub
# Colab ships Weights & Biases; Ultralytics would log to it, and it rejects our
# run folder. Removed BEFORE ultralytics is first imported - switching it off
# afterwards is too late.
!pip -q uninstall -y wandb
import os
# Colab's PyTorch is newer than the add-on's and refuses to load YOLO model
# files by default ("Weights only load failed"). These are Ultralytics' own files.
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to T4 GPU")

In [ ]:
# D-Fire, already in YOLO format (the Kaggle copy the D-Fire authors link to).
# If Kaggle asks you to sign in: kaggle.com -> Settings -> Create New Token,
# then upload the kaggle.json it gives you with the folder icon on the left.
import kagglehub
root = kagglehub.dataset_download("sayedgamal99/smoke-fire-detection-yolo")
print(root)
!find "$root" -maxdepth 3 -type d | head -30
!find "$root" -name "*.yaml" | head

In [ ]:
# Find the train / val / test folders whatever the archive calls them, and
# read the class names from the dataset's own yaml (D-Fire: 0 smoke, 1 fire).
import glob, os, yaml
def split_dir(*names):
    for n in names:
        hits = [p for p in glob.glob(f"{root}/**/{n}/images", recursive=True)]
        if hits:
            return hits[0]
    return None
train = split_dir("train"); val = split_dir("val", "valid"); test = split_dir("test")
names = None
for y in glob.glob(f"{root}/**/*.yaml", recursive=True):
    names = (yaml.safe_load(open(y)) or {}).get("names") or names
names = names or ["smoke", "fire"]
assert train and val, "could not find train/val image folders - look at the listing above"
cfg = {"train": train, "val": val, "test": test or val, "names": names}
yaml.safe_dump(cfg, open("/content/dfire.yaml", "w"))
print(open("/content/dfire.yaml").read())
for k in ("train", "val", "test"):
    print(k, len(os.listdir(cfg[k])), "images")

In [ ]:
# Same small model the add-on ships (fast enough on a Raspberry Pi / HA Green).
# Fixed seed so a re-run gives the same result.
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
model.train(data="/content/dfire.yaml", epochs=60, imgsz=640, batch=32,
            patience=15, seed=0, deterministic=True, workers=2,
            project="/content/runs", name="dfire", exist_ok=True)

In [ ]:
# Score BOTH saved checkpoints on the held-out TEST photos, separately from
# training. (In an earlier FiremeX run the scores printed during training were
# wrong and the worse checkpoint was shipped - so we check, not trust.)
results = {}
for which in ("best", "last"):
    m = YOLO(f"/content/runs/dfire/weights/{which}.pt")
    r = m.val(data="/content/dfire.yaml", split="test", imgsz=640, batch=32, plots=False)
    per = {m.names[i]: (round(float(r.box.p[k]), 3), round(float(r.box.r[k]), 3))
           for k, i in enumerate(r.box.ap_class_index)}
    results[which] = (float(r.box.map50), per)
    print(f"{which}.pt  mAP50={r.box.map50:.3f}  (precision, recall) per class: {per}")
chosen = max(results, key=lambda w: results[w][0])
print("\nUsing", chosen + ".pt")

In [ ]:
import shutil
from google.colab import files
shutil.copy(f"/content/runs/dfire/weights/{chosen}.pt", "/content/fire_model_dfire.pt")
!sha256sum /content/fire_model_dfire.pt
files.download("/content/fire_model_dfire.pt")